In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus(n_gpus=1)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import plotnine as gg
from essential.flow import FlowMatchingEstimator

In [ ]:
import sys

sys.path.append("/workspace/experiments/12012025_flows")

# from flow_matching_config import get_config

from dynamic_config import get_config

config = get_config()
config.training.n_epochs = 250
config.training.batch_size = 128
config.estimator.expression_type = "logmedian"
config.training.learning_rate = 1e-3

print(config)

In [ ]:
adata = sc.read_h5ad(config.processing.adata_path)
if config.processing.rt_bc != "all":
    adata = adata[adata.obs["rt_bc"] == config.processing.rt_bc].copy()
if config.processing.consolidated_cluster != "all":
    adata = adata[
        adata.obs["consolidated_cluster"] == config.processing.consolidated_cluster
    ].copy()

sc.pp.filter_genes(adata, min_cells=10)

adata.obs["rt_bc"].value_counts()

In [ ]:
# sc.pp.filter_genes(adata, min_cells=5000)
# adata

In [ ]:
estimator = FlowMatchingEstimator(adata, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())

In [ ]:
estimator.val_indices_perturbed

In [ ]:
estimator.epoch_history_df["val_reco_loss"].plot()

In [ ]:
estimator.epoch_history_df["train_loss"].plot()

In [ ]:
estimator.step_history_df["loss"].plot()
# plt.yscale("log")

In [ ]:
# from essential.ode import ODEstimator


# def get_interaction_matrix(self, return_square=True, delta=None):
#     if self.state is None:
#         raise RuntimeError("Model has not been trained yet. Please call .fit() first.")

#     processed_Amat = self.model.apply({"params": self.state.params}, method=self.model.get_Amat)
#     if processed_Amat.shape == (self.n_genes, self.n_genes):
#         Amat_ = pd.DataFrame(
#             processed_Amat, index=self.adata.var_names, columns=self.adata.var_names
#         )
#     elif processed_Amat.shape == (self.n_genes, self.n_perturbations):
#         Amat_ = pd.DataFrame(
#             processed_Amat, index=self.adata.var_names, columns=self.gene_perturbations
#         )
#     else:
#         raise ValueError(f"Invalid Amat shape: {processed_Amat.shape}")
#     return ODEstimator.process_interaction_matrix(Amat_, return_square=return_square, delta=delta)

In [ ]:
a_mat = estimator.get_interaction_matrix()

In [ ]:
from essential.utils import compute_topk_precision_metrics


delta = config.estimator.get("delta", 0.125)
processed_a_mat = FlowMatchingEstimator.process_interaction_matrix(
    a_mat, return_square=False, delta=delta
)
topk_precision_df = compute_topk_precision_metrics(processed_a_mat, "compute")

In [ ]:
a_mat_ = a_mat.values.flatten()

In [ ]:
plt.hist(a_mat_, bins=100)
plt.yscale("log")
plt.show()

In [ ]:
for i, row in topk_precision_df.loc[lambda x: x["model"] == "compute"].head(100).iterrows():
    print(row["regulator_gene_"], row["target_gene_"])

In [ ]:
plot_df = topk_precision_df.query("type == 'offdiag'")

fig = (
    gg.ggplot(plot_df, gg.aes(x="topk", y="n_tp", color="model"))
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(6.5, 4))
    + gg.labs(x="top K predicted interactions", y="# of hits in RegulonDB")
)
fig